# Práctica 03. Análisis Estadistico de señales

**Estudiante:** Juliana Sarmiento Arciniegas

**Identificación:** 1037667898

 Funciones en Python:

*   **mean:** Calcula el promedio de un conjunto de datos (np.mean())
*   **var:** Calcula la varianza de un conjunto de datos, es decir mide cuánto se dispersan los datos respecto a la media.
*   **std:** Calcula la desviación estandar (raiz cuadrada de la varianza) y arroja valores más faciles de interpretar que la varianza

In [1]:
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from scipy.signal import find_peaks #ayudará más adelnate a buscar picos de señales
import pandas as pd
from scipy.stats import shapiro #Prueba de Shapiro
from scipy.stats import levene #Prueba de homocedasticidad
from scipy.stats import mannwhitneyu #Prueba de Mann Whitney U
from statsmodels.tsa.stattools import adfuller #Prueba de Augmented Dickey-Fuller

## Exportar los datos de interes para ECG

In [2]:
# Cargar archivo .mat
mat = sio.loadmat("signals.mat")

# Ver qué variables contiene para saber cuales exportar
print(mat.keys())

#verificar el tamaño del arreglo
raw = mat['ECG_asRecording']
print(raw.shape)

dict_keys(['__header__', '__version__', '__globals__', 'Fs', 'ECG_asRecording', 'ECG_filtered', 'EMG_filtered1', 'EMG_filtered2', 'EMG_asRecording1', 'EMG_asRecording2'])
(1, 30720)


Se hará uso de la funcion np.squeeze ya que esta permite pasar de un vector 2D, como se observa en el tamaño del vector ECG_asRecording, a un arreglo plano de una dimensión. Esto es importante para realizar operaciones matemáticas a lo largo del informe.

In [3]:
fs = float(np.squeeze(mat['Fs']))                    # frecuencia de muestreo
ecg_original = np.squeeze(mat['ECG_asRecording'])    # senal original, tal cual se registro
ecg_filtrada = np.squeeze(mat['ECG_filtered'])       # senal filtrada

print("Frecuencia de muestreo:", fs, "Hz")
print("Tamano ECG cruda:", ecg_original.shape)
print("Tamano ECG filtrada:", ecg_filtrada.shape)

Frecuencia de muestreo: 1024.0 Hz
Tamano ECG cruda: (30720,)
Tamano ECG filtrada: (30720,)


Ya teniendo un tamaño de (30720,) es más fácil continuar con las operaciones a lo largo del taller.

## Creación de la funcion RMS

### Valor cuadrático medio (RMS)

$$x_{RMS} = \left[\frac{1}{N}\sum_{i=1}^{N} x(i)^2\right]^{1/2}$$

In [4]:
def rms(x):
    x = np.asarray(x, dtype=float)   # asarray -> asegura que la entrada sea un arreglo de numpy
    return np.sqrt(np.mean(x**2))

# prueba rapida con una senoidal de amplitud conocida (RMS teorico = A/sqrt(2))
t_prueba = np.linspace(0, 1, 1000, endpoint=False)
seno_prueba = 5*np.sin(2*np.pi*10*t_prueba)
print("RMS teorico:  ", 5/np.sqrt(2))
print("RMS calculado:", rms(seno_prueba))

RMS teorico:   3.5355339059327373
RMS calculado: 3.5355339059327378


## Tiempo de duración de las señales

In [ ]:
muestras_original = len(ecg_original)#muestras del vector original
muestras_filtrada = len(ecg_filtrada)#muestras del vector filtrado

#calcular el tiempo
tiempo_original = muestras_original/fs # tiempo total = N * (1/fs)
tiempo_filtrado = muestras_filtrada/fs
print("Duracion senal original :", tiempo_original, "s")
print("Duracion senal filtrada :", tiempo_filtrado, "s")

# vector de tiempo (comun a ambas senales, mismo numero de muestras)
tiempo = np.arange(muestras_filtrada)/fs #convertir a un arreglo numpy
print("Vector de tiempo (primeras 5 muestras):", tiempo[:5])

Duracion senal original : 30.0 s
Duracion senal filtrada : 30.0 s
Vector de tiempo (primeras 5 muestras): [0.         0.00097656 0.00195312 0.00292969 0.00390625]


## Comparativa entre señal filtrada y sin filtrar

Se realiza un recorte en la señal para visualizar solo el comportamiento de las señales en los dos primeros segundos.